In [33]:
import json
import os, sys
from google.protobuf import json_format, timestamp_pb2
import pandas as pd

p = os.getcwd()
while p != os.path.dirname(p):
    if os.path.isdir(os.path.join(p, "cpp")):
        sys.path.insert(0, os.path.join(p, "pyhexz/src"))
        break
    p = os.path.dirname(p)
else:
    raise IOError("Source root not found. This notebook will not work.")

from pyhexz import nbench_pb2

In [14]:
results = []
with open(os.path.join(os.getenv("HOME"), "git/github.com/dnswlt/hexz/stats/nbench.jsonl")) as f_in:
    for line in f_in:
        msg = nbench_pb2.BenchmarkResult()
        json_format.Parse(line, msg)
        results.append(msg)
print(f"Read {len(results)} entries.")

Read 37 entries.


In [50]:
def mkey(p: nbench_pb2.BenchmarkResult.PlayerResult) -> str:
    return f"{p.model_key.name}:{p.model_key.checkpoint}"

class Table:
    def __init__(self, results):
        self.results = results
        self.players = sorted(set(mkey(r.p1_result) for r in results).union(set(mkey(r.p2_result) for r in results)))
        self.idx = dict(zip(self.players, range(len(self.players))))
        self.points = [[0] * len(self.players) for _ in range(len(self.players))]
        for r in self.results:
            p1 = mkey(r.p1_result)
            p2 = mkey(r.p2_result)
            draws = r.games - (r.p1_result.wins + r.p2_result.wins)
            self.points[self.idx[p1]][self.idx[p2]] += r.p1_result.wins + draws / 2
            self.points[self.idx[p2]][self.idx[p1]] += r.p2_result.wins + draws / 2
    
    def win_rates(self):
        wr = [[0] * len(self.players) for _ in range(len(self.players))]
        for i in range(len(wr)):
            for j in range(len(wr[0])):
                s = self.points[i][j] + self.points[j][i]
                if s > 0:
                    wr[i][j] = self.points[i][j] / s
        return wr

    def elo_scores(self, k=32, scaling_factor=400, initial_score=1500):
        elos = {p: initial_score for p in self.players}
        
        for r in self.results:
            if r.games == 0:
                continue  # should never happen
            p1 = mkey(r.p1_result)
            p2 = mkey(r.p2_result)
            # Current Elo ratings
            elo_p1 = elos[p1]
            elo_p2 = elos[p2]
            
            # Expected scores
            expected_p1 = r.games / (1 + 10 ** ((elo_p2 - elo_p1) / scaling_factor))
            expected_p2 = r.games - expected_p1
            
            # Actual scores
            draws = r.games - (r.p1_result.wins + r.p2_result.wins)
            actual_p1 = r.p1_result.wins + draws / 2
            actual_p2 = r.p2_result.wins + draws / 2
            
            # Update Elo scores
            elos[p1] += k * (actual_p1 - expected_p1)
            elos[p2] += k * (actual_p2 - expected_p2)

        return elos    
        
        
        


In [53]:
t = Table(results)
t.elo_scores()

{'res10:20': 1404.0,
 'res10:30': 1479.1178672167414,
 'res10:50': 1407.9198856572252,
 'res10:51': 1483.3898751427528,
 'res10:52': 1477.6561404198148,
 'res10:53': 1584.9847394587275,
 'res10:54': 1416.3089669682995,
 'res10:55': 1424.695249393013,
 'res10:56': 1680.3716323463768,
 'res10:57': 1229.988037182029,
 'res10:58': 1477.3294920217415,
 'res10:59': 1784.3390173217201,
 'res10:60': 1614.2496419088418,
 'res10:61': 1336.2793616513673,
 'res10:62': 1389.9225339386956,
 'res10:63': 1809.4475593726545}

In [47]:
r1 = 2100
r2 = 1500
scaling_factor = 400
e1 = 1 / (1 + 10 ** ((r2 - r1) / scaling_factor))
e2 = 1 / (1 + 10 ** ((r1 - r2) / scaling_factor))
e1, e2, e1+e2

(0.9693465699682844, 0.030653430031715508, 0.9999999999999999)

In [61]:
from random import shuffle
ps = [i for i in range(50, 64)]
shuffle(ps)
for i in range(0, len(ps)-2, 2):
    print(f"bash scripts/nbench2.sh {ps[i]} {ps[i+1]}")

bash scripts/nbench2.sh 52 60
bash scripts/nbench2.sh 53 50
bash scripts/nbench2.sh 57 61
bash scripts/nbench2.sh 62 51
bash scripts/nbench2.sh 58 55
bash scripts/nbench2.sh 54 63


In [41]:
t = Table(results)
pd.DataFrame(t.win_rates(), columns=t.players, index=t.players) * 100

,res10:20,res10:30,res10:50,res10:51,res10:52,res10:53,res10:54,res10:55,res10:56,res10:57,res10:58,res10:59,res10:60,res10:61,res10:62,res10:63
res10:20,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,20.00
res10:30,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,30.00
res10:50,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,42.00
res10:51,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,51.25
res10:52,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,48.75
res10:53,0.0,0.0,0.0,0.00,0.00,0.00,80.0,60.00,70.00,80.00,60.00,60.0,70.00,80.00,0.0,56.25
res10:54,0.0,0.0,0.0,0.00,0.00,20.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,40.00
res10:55,0.0,0.0,0.0,0.00,0.00,40.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,46.25
res10:56,0.0,0.0,0.0,0.00,0.00,30.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,68.75
res10:57,0.0,0.0,0.0,0.00,0.00,20.00,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,36.25
